# Can we predict `passed_quiz`?
Target: whether the student passes the quiz closing a session. Follows `checklist.md`.

**Design notes**
- There is no date/time column, so the "later rows" rule can't be applied; no claim of temporal validity is made.
- `student_key` repeats (about 7 sessions per student), so every split (test set and CV folds) holds **whole students**.
- `student_key` is an identifier and is dropped. `enrol_year`, `school_code` and `prior_score` (placement test) are known before the session. `session_minutes` and `hints_used` are recorded *during* the session, so the model applies once the session's work is done, just before the quiz (not at session start). This is a scope assumption.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, log_loss
df = pd.read_csv('data.csv')
print(df.shape, df.passed_quiz.mean())

## 1. Test set first (whole students, 20%)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, te_idx = next(gss.split(df, groups=df.student_key))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert not set(train.student_key) & set(test.student_key)
print(len(train), len(test), train.passed_quiz.mean(), test.passed_quiz.mean())
train.describe().T  # exploration uses training rows only

In [ ]:
FEATURES = ['enrol_year','school_code','prior_score','session_minutes','hints_used']
X, y, g = train[FEATURES], train.passed_quiz, train.student_key
train.groupby('passed_quiz')[FEATURES].mean()

## 2. Candidate models (named before any scoring)
1. `baseline`: predicts the training class prior (trivial).
2. `logreg`: standardised numeric features + one-hot `school_code`, L2 logistic regression, C in {0.01, 0.1, 1}.
3. `hgb`: histogram gradient boosting, grid max_depth in {2, 3} x learning_rate in {0.03, 0.1}.

Selection metric: ROC AUC. The decision threshold is then chosen on out-of-fold predictions to maximise balanced accuracy.
Preprocessing lives inside the pipelines, so it is fitted within each CV fold.

In [ ]:
num = ['enrol_year','prior_score','session_minutes','hints_used']
def logreg(C):
    pre = ColumnTransformer([('n', StandardScaler(), num), ('s', OneHotEncoder(handle_unknown='ignore'), ['school_code'])])
    return Pipeline([('pre', pre), ('m', LogisticRegression(C=C, max_iter=1000))])
def hgb(d, lr):
    return Pipeline([('m', HistGradientBoostingClassifier(max_depth=d, learning_rate=lr, max_iter=200, random_state=0))])
candidates = {'baseline': Pipeline([('m', DummyClassifier(strategy='prior'))])}
for c in (0.01,0.1,1): candidates[f'logreg_C{c}'] = logreg(c)
for d in (2,3):
    for lr in (0.03,0.1): candidates[f'hgb_d{d}_lr{lr}'] = hgb(d, lr)
list(candidates)

## 3. Cross-validation on training rows (5 folds, grouped by student)

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=1)
folds = list(cv.split(X, y, g))
oof, rows = {}, []
for name, est in candidates.items():
    p = cross_val_predict(est, X, y, cv=folds, method='predict_proba')[:,1]
    oof[name] = p
    fold_auc = [roc_auc_score(y.iloc[v], p[v]) for _, v in folds]
    rows.append((name, np.mean(fold_auc), np.std(fold_auc), log_loss(y, p)))
cvres = pd.DataFrame(rows, columns=['model','cv_auc','auc_sd_across_folds','logloss']).sort_values('cv_auc', ascending=False)
cvres

In [ ]:
best = cvres.iloc[0].model
p = oof[best]
ths = np.linspace(0.1, 0.9, 81)
bal = [balanced_accuracy_score(y, p >= t) for t in ths]
threshold = float(ths[int(np.argmax(bal))])
print(best, 'threshold', threshold, 'cv balanced acc', max(bal))
print('baseline cv balanced acc', balanced_accuracy_score(y, oof['baseline']>=0.5))

## 4. Choice (fixed from CV only, before test scoring)
Chosen model = the highest CV-AUC model above (`best`), with the CV-chosen threshold. Nothing changes after this point.

## 5. Refit on all training rows, score the test set once

In [ ]:
final = candidates[best].fit(X, y)
pt = final.predict_proba(test[FEATURES])[:,1]
yt = test.passed_quiz
base = candidates['baseline'].fit(X, y).predict_proba(test[FEATURES])[:,1]
rng = np.random.default_rng(0)
studs = test.student_key.unique(); idx = test.groupby('student_key').indices
aucs = []
for _ in range(500):
    s = rng.choice(studs, len(studs))
    ii = np.concatenate([idx[k] for k in s])
    aucs.append(roc_auc_score(yt.iloc[ii], pt[ii]))
print(f'HEADLINE test AUC {roc_auc_score(yt, pt):.3f}  (95% student-bootstrap CI {np.percentile(aucs,2.5):.3f}-{np.percentile(aucs,97.5):.3f})')
print(f'test balanced acc @ {threshold:.2f}: {balanced_accuracy_score(yt, pt>=threshold):.3f}; accuracy {accuracy_score(yt, pt>=threshold):.3f}')
print(f'baseline test AUC {roc_auc_score(yt, base):.3f}; always-majority accuracy {1-yt.mean():.3f}')

## Conclusion
The headline is the test AUC above, scored once. The CV table drove all choices. Caveats: no timestamp, so generalisation to future sessions is untested; `session_minutes`/`hints_used` exist only once the session is underway.